# Role-Conditioned DPO Training
Upload this notebook to Google Colab. Make sure to select a **GPU runtime** (Runtime > Change runtime type > T4 GPU or better).

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers>=4.40 datasets>=2.18 trl>=0.8 peft>=0.10 accelerate>=0.28 bitsandbytes>=0.43 wandb

In [ ]:
# Step 2: Upload your DPO training data
# Option A: Upload from local machine
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print('Upload dpo_pairs_train.jsonl:')
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f'data/{name}', 'wb') as f:
        f.write(content)
    print(f'  Saved data/{name}')

print('\nUpload dpo_pairs_eval.jsonl:')
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f'data/{name}', 'wb') as f:
        f.write(content)
    print(f'  Saved data/{name}')

In [ ]:
# Step 2 (Alternative): Mount Google Drive instead of uploading
# Uncomment if you prefer to put data in Drive

# from google.colab import drive
# drive.mount('/content/drive')
# # Then set paths like:
# # TRAIN_PATH = '/content/drive/MyDrive/cs288/dpo_pairs_train.jsonl'
# # EVAL_PATH = '/content/drive/MyDrive/cs288/dpo_pairs_eval.jsonl'

In [ ]:
# Step 3: Verify data
import json

TRAIN_PATH = 'data/dpo_pairs_train.jsonl'
EVAL_PATH = 'data/dpo_pairs_eval.jsonl'

def count_lines(path):
    with open(path) as f:
        return sum(1 for _ in f)

print(f'Training pairs: {count_lines(TRAIN_PATH)}')
print(f'Eval pairs: {count_lines(EVAL_PATH)}')

# Preview one example
with open(TRAIN_PATH) as f:
    example = json.loads(f.readline())
print(f'\nExample prompt (first 200 chars):\n{example["prompt"][:200]}')
print(f'\nChosen (first 150 chars):\n{example["chosen"][:150]}')
print(f'\nRejected (first 150 chars):\n{example["rejected"][:150]}')

In [ ]:
# Step 4: Load model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
# Other options:
# MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'  # requires HF access
# MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    quantization_config=bnb_config,
)
print('Model loaded.')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Step 5: Prepare datasets
from datasets import Dataset

def load_preference_data(path):
    records = []
    with open(path) as f:
        for line in f:
            data = json.loads(line)
            records.append({
                'prompt': data['prompt'],
                'chosen': data['chosen'],
                'rejected': data['rejected'],
            })
    return Dataset.from_list(records)

train_dataset = load_preference_data(TRAIN_PATH)
eval_dataset = load_preference_data(EVAL_PATH)
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
# Step 6: Configure LoRA and DPO training
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)

# DPO training config
# Change loss_type to 'ipo' for IPO variant
training_args = DPOConfig(
    output_dir='checkpoints/role-conditioned-dpo',
    loss_type='sigmoid',  # 'sigmoid' = DPO, 'ipo' = IPO
    beta=0.1,
    learning_rate=5e-7,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_length=1024,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    bf16=True,
    report_to='none',  # set to 'wandb' if you want logging
    gradient_checkpointing=True,  # saves memory on Colab
)

print('Config ready.')
print(f'  Loss: {training_args.loss_type}')
print(f'  Beta: {training_args.beta}')
print(f'  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')

In [ ]:
# Step 7: Train!
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print('Starting DPO training...')
trainer.train()
print('Training complete!')

In [ ]:
# Step 8: Save model
OUTPUT_DIR = 'checkpoints/role-conditioned-dpo-final'
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

# Download the LoRA adapter weights
!zip -r role_conditioned_dpo_lora.zip {OUTPUT_DIR}
files.download('role_conditioned_dpo_lora.zip')

In [ ]:
# Step 9: Quick inference test
from peft import PeftModel

test_prompt = """You are a middle manager in engineering, who outranks the person they are speaking with. You have 1-5 years at the company.

Interaction context:
You are assigning a task to someone. Describe the task clearly, set expectations, and provide any necessary context.

Situation: The team needs to migrate the authentication service from the legacy monolith to a microservice architecture before the Q3 deadline.
Background: The migration has been planned for months but keeps getting deprioritized. The engineer you're assigning this to is capable but has a full plate.

Respond appropriately given your role and the situation."""

inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Generated response:')
print(response)